# Penalty frictional mortar contact condition

This notebook generates `custom_conditions/penalty_frictional_mortar_contact_condition.cpp`, the local
left- and right-hand sides of `PenaltyMethodFrictionalMortarContactCondition<TDim, TNumNodes, TNormalVariation, TNumNodesMaster>`,
the **penalty frictional** mortar contact condition (no multiplier DoFs). Theory: thesis §4.3.4 and the
[Frictional contact](https://kratosmultiphysics.github.io/Kratos/pages/Applications/Contact_Structural_Mechanics_Application/Theory/Frictional_Contact.html)
page of the documentation.

## Formulation

### Penalty regularisation of Coulomb's law

Without multipliers the normal pressure is approximated by $\varepsilon_n\, g_n$ (thesis eq. 4.57-4.58) and the
Coulomb law (thesis eq. 4.45) is regularised with a tangential penalty $\varepsilon_\tau = \kappa\,\varepsilon_n$:
in **stick** the tangential traction is $\varepsilon_\tau\, \tilde{\mathbf{u}}_\tau$ and in **slip** it is the Coulomb
threshold

$$\mathscr{F} = -\mu\, \varepsilon_n\, \tilde{g}_n\, \boldsymbol{\tau}$$

where, since no multiplier is available, the tangent direction is taken from the slip increment itself,
$\boldsymbol{\tau} = \tilde{\mathbf{u}}_\tau / \Vert \tilde{\mathbf{u}}_\tau \Vert$ (`ComputeTangentMatrixSlip`, nodal `TANGENT_XI`
computed from `WEIGHTED_SLIP` by `ComputePenaltyFrictionalActiveSet`). The corresponding algebraic system is
thesis eq. 4.71a: the contact contributions are added directly to the displacement blocks of the master and
active slave nodes, split into slip and stick restrictions, and inactive nodes contribute nothing.

### Discrete gap and slip

With the mortar operators $\mathbf{D}$, $\mathbf{M}$ (thesis eq. 4.29) the nodal weighted gap is
$\tilde{g}_{n,j} = -\mathbf{n}_j \cdot (\mathbf{D}\mathbf{x}^1 - \mathbf{M}\mathbf{x}^2)_j$ (positive when open) and the
objective nodal slip increment is (thesis eq. 4.69b)

$$\tilde{\mathbf{u}}_{\tau,j} = \boldsymbol{\tau}_j \left[ \left( \mathbf{D}^{t+\Delta t} - \mathbf{D}^{t} \right) \mathbf{x}^{1} - \left( \mathbf{M}^{t+\Delta t} - \mathbf{M}^{t} \right) \mathbf{x}^{2} \right]_j$$

built from the mortar operators of the previous converged step (`DOperatorold`, `MOperatorold`). The penalty
condition has no multiplier equation, so it does not need the objective / non-objective switch of the ALM
condition: only the objective slip enters the functional (the `TangentSlipNonObjective` symbols are
available in `SymbolSet` but unused here), and the C++ `WEIGHTED_SLIP` of the penalty conditions is
always the objective one.

### The three generated branches

Per slave node $j$, with $\mathcal{D}_j$ the `DynamicFactor`, $p_{n,j} = \varepsilon_j\,\tilde{g}_{n,j}$ and the
test-function quantities $X_w = -\delta X$ (`NormalwGap`, `TangentwSlipObjective`):

| branch | $\mathcal{R}_j$ |
|---|---|
| inactive | $0$ |
| active slip | $\mathcal{D}_j \left[ p_{n,j}\, \tilde{g}_{n,j,w} + \left( -\mu_j\, p_{n,j}\, \boldsymbol{\tau}_j \right) \cdot \tilde{\mathbf{u}}_{\tau,j,w} \right]$ |
| active stick | $\mathcal{D}_j \left[ p_{n,j}\, \tilde{g}_{n,j,w} + \left( \kappa\, \varepsilon_j\, \tilde{\mathbf{u}}_{\tau,j} \right) \cdot \tilde{\mathbf{u}}_{\tau,j,w} \right]$ |

i.e. the virtual work of the normal penalty traction and of the tangential (Coulomb or penalty) traction.
The run-time dispatch is, per node, `if (r_geometry[i].IsNot(ACTIVE)) {...} else if (r_geometry[i].Is(SLIP)) {...} else {...}`.

## From the functional to the generated C++

All the mechanics of the generation live in `../mortar_condition_generator.py` (see also the
[Automatic differentiation](https://kratosmultiphysics.github.io/Kratos/pages/Applications/Contact_Structural_Mechanics_Application/Theory/Automatic_Differentiation.html)
page, thesis Appendix C):

1. **Symbols** (`SymbolSet`): the nodal unknowns `u1`, `u2` (displacements of the slave and master nodes),
   the multipliers, the test functions `w1`, `w2`, `wLM`, the reference coordinates `X1`, `X2`, the nodal
   normals `NormalSlave`, the mortar operators `DOperator`, `MOperator` and the parameters. The current
   coordinates are $\mathbf{x}^{(i)} = \mathbf{X}^{(i)} + \mathbf{u}^{(i)}$ and the nodal **weighted gap**
   (thesis eq. 4.31) is
   $$\tilde{g}_{n,j} = -\,\mathbf{n}_j \cdot \left( \mathbf{D}\, \mathbf{x}^{(1)} - \mathbf{M}\, \mathbf{x}^{(2)} \right)_j$$
   which is **positive for an open gap** and negative for penetration (`NormalGap` / `WEIGHTED_GAP`).
2. **AD exceptions** (thesis §C.3.1): $\mathbf{D}$, $\mathbf{M}$ and, when `TNormalVariation` is `true`, $\mathbf{n}$
   are not expressed in terms of the displacements. They are declared *undefined functions of the DoFs*
   (`DefineDofDependencyMatrix`), so that the chain rule produces unevaluated derivatives that are mapped to
   the arrays computed at run time by `DerivativesUtilities`:

   | symbolic node | C++ |
   |---|---|
   | `DOperator_i_j(u...)` | `DOperator(i,j)` |
   | `Derivative(DOperator_i_j(u...), u_k)` | `DeltaDOperator[k](i,j)` |
   | `Derivative(MOperator_i_j(u...), u_k)` | `DeltaMOperator[k](i,j)` |
   | `Derivative(NormalSlave_i_j(u...), u_k)` | `DeltaNormalSlave[k](i,j)` (normal variation only) |

   The index `k` runs over the slave displacement DoFs first and then the master ones, the ordering used by
   `MortarOperatorWithDerivatives`.
3. **Differentiation**: for every slave node $j$ and every active-set branch the functional $\mathcal{R}_j$
   returned by the function below is differentiated: $\mathbf{r} = \partial \mathcal{R} / \partial \mathbf{w}$
   (local RHS) and $\mathbf{K} = -\partial \mathbf{r} / \partial \mathbf{d}$ (local LHS), with the DoF vector
   $\mathbf{d} = [\mathbf{u}^{(2)}, \mathbf{u}^{(1)}, \boldsymbol{\lambda}]$ ordered *master, slave, multiplier* exactly as
   `GetDofList`. This is the Kratos convention $\mathbf{K}\,\Delta\mathbf{d} = \mathbf{r}$.
4. **Printing**: the derivative nodes are replaced by plain symbols, `sympy.cse` collects the common factors
   (`clhs*`, `crhs*`) and `sympy.ccode` prints C++; only the non-zero entries are emitted, accumulated with `+=`.
5. **Assembly of the file**: one `CalculateLocalLHS` specialisation per geometry pair (`2D2N`, `3D3N`, `3D4N`,
   `3D3N4N`, `3D4N3N`) and per `TNormalVariation` value; the RHS does **not** depend on the derivatives of the
   normal, so `StaticCalculateLocalRHS` is generated only for `TNormalVariation = false` and the `true`
   specialisation forwards to it. The bodies are substituted into the `*_template.cpp` of this folder at the
   `// replace_lhs` / `// replace_rhs` markers and the result is written once.

**Sign convention of the test-function quantities.** Every `<quantity>w` symbol (`NormalwGap`, `TangentwSlip*`)
is *minus* the variation of the quantity in the direction of the test functions, $X_w = -\delta X$: with the
gap defined as above, `NormalwGap = +n.(D w1 - M w2)`, so that the virtual work of a traction $\mathbf{t}$
is written $\mathbf{t} \cdot X_w$ and the residual is $-\delta\Pi$ (the force acting on the bodies).

## How to run this notebook

* **Requirements**: Python 3 and `sympy` (any modern version, tested with 1.14). A compiled Kratos is *not*
  needed: the shared module `../mortar_condition_generator.py` imports `custom_sympy_fe_utilities.py` and the
  core `sympy_fe_utilities.py` directly from the source tree.
* **Interactively**: open it with Jupyter from this folder and run all cells.
* **Headless** (no Jupyter installed): `python3 ../run_notebook.py <this notebook>` executes the code cells
  with the standard library only.
* The equivalent command-line script `generate_*.py` in this folder contains the *same* functional and
  generation call; keep both in sync when the formulation changes.

The output is written directly into `custom_conditions/` (overwriting the committed file). The generation of
the five geometries and the two normal-variation flags takes from a few minutes (frictionless) to about an
hour (ALM frictional). Restrict `COMBINATIONS` / `NORMAL_VARIATIONS` in the configuration cell for a quick
test, or set `OUTPUT_DIR` to a scratch folder.

In [ ]:
import os
import sys

# The shared generator module lives one folder up (automatic_differentiation/)
sys.path.insert(0, os.path.abspath(".."))
import mortar_condition_generator as generator

print("sympy", generator.sympy.__version__)

## Symbols of `SymbolSet` used by the functional

| symbol | meaning | C++ counterpart |
|---|---|---|
| `s.NormalGap[j]`, `s.NormalwGap[j]` | $\tilde{g}_{n,j}$ and $-\delta\tilde{g}_{n,j}$ | `WEIGHTED_GAP` |
| `s.NormalSlave.row(j)`, `s.TangentSlave.row(j)` | $\mathbf{n}_j$, $\boldsymbol{\tau}_j$ | `NORMAL`, `TANGENT_XI` |
| `s.TangentSlipObjective.row(j)`, `s.TangentwSlipObjective.row(j)` | $\tilde{\mathbf{u}}_{\tau,j}$ and $-\delta\tilde{\mathbf{u}}_{\tau,j}$ | `WEIGHTED_SLIP` |
| `s.PenaltyParameter[j]`, `s.TangentFactor`, `s.mu[j]`, `s.DynamicFactor[j]` | $\varepsilon_j$, $\kappa$, $\mu_j$, $\mathcal{D}_j$ | `INITIAL_PENALTY`, `TANGENT_FACTOR`, `FRICTION_COEFFICIENT`, `DYNAMIC_FACTOR` |

The branch identifiers passed to the functional are `inactive`, `slip` and `stick`.

## The functional

This is the physics of the condition; it is the only family-specific input of the generator.

In [ ]:
def penalty_frictional_functional(s, node, branch):
    """Galerkin functional of one slave node in one active-set branch (thesis eqs. 4.57-4.58, 4.71).

    ``branch`` is one of ``inactive``, ``slip``, ``stick``. Only the objective slip is used (there is no
    multiplier equation, so the objective/non-objective switch of the ALM condition is not needed).
    """
    rv_galerkin = 0
    if branch == "inactive":
        rv_galerkin += 0
    else:
        # Normal penalty traction
        augmented_normal_contact_pressure = s.PenaltyParameter[node] * s.NormalGap[node]
        rv_galerkin += s.DynamicFactor[node] * augmented_normal_contact_pressure * s.NormalwGap[node]

        if branch == "slip":
            # Coulomb traction -mu * p_n * tau
            augmented_tangent_contact_pressure = - s.mu[node] * augmented_normal_contact_pressure * s.TangentSlave.row(node)
        else:
            # Tangential penalty traction
            augmented_tangent_contact_pressure = s.TangentFactor * s.PenaltyParameter[node] * s.TangentSlipObjective.row(node)
        rv_galerkin += s.DynamicFactor[node] * augmented_tangent_contact_pressure.dot(s.TangentwSlipObjective.row(node))

    return rv_galerkin

## Configuration

`COMBINATIONS` holds the `(dimension, slave nodes, master nodes)` triplets to generate and
`NORMAL_VARIATIONS` the values of `TNormalVariation`. The output goes to `custom_conditions/` by default.

In [ ]:
COMBINATIONS = generator.DEFAULT_COMBINATIONS   # ((2, 2, 2), (3, 3, 3), (3, 4, 4), (3, 3, 4), (3, 4, 3))
NORMAL_VARIATIONS = (False, True)
TEMPLATE_DIR, OUTPUT_DIR = generator.DefaultDirectories(os.getcwd())
print("template folder:", TEMPLATE_DIR)
print("output folder  :", OUTPUT_DIR)

## Generation

Every branch reports the size of the local system and the number of non-zero entries. The generated
code is checked for symbolic leftovers before the file is written.

In [ ]:
output_path = generator.Generate(generator.PENALTY_FRICTIONAL, penalty_frictional_functional, TEMPLATE_DIR, OUTPUT_DIR, COMBINATIONS, NORMAL_VARIATIONS)

## Check of the generated file

The file must contain, for each of the geometries and normal-variation flags requested, one
`CalculateLocalLHS` specialisation and one `StaticCalculateLocalRHS` specialisation (a full body for
`false`, a forwarder for `true`).

In [ ]:
import re

with open(output_path) as generated_file:
    generated = generated_file.read()

lhs_specialisations = re.findall(r"^void PenaltyMethodFrictionalMortarContactCondition<(\d+),\s*(\d+), (true|false), (\d+)>::CalculateLocalLHS\(", generated, re.MULTILINE)
rhs_specialisations = re.findall(r"^void PenaltyMethodFrictionalMortarContactCondition<(\d+),\s*(\d+), (true|false), (\d+)>::StaticCalculateLocalRHS\(", generated, re.MULTILINE)
expected = len(COMBINATIONS) * len(NORMAL_VARIATIONS)
print("{} lines, {} LHS and {} RHS specialisations (expected {} each)".format(generated.count("\n"), len(lhs_specialisations), len(rhs_specialisations), expected))
assert len(lhs_specialisations) == expected and len(rhs_specialisations) == expected
assert "Derivative(" not in generated and "//subsvar_" not in generated